In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIXED #####
# The Crud module is actual Python module that Dash imports and uses to talk to MongoDB.
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
# instead of using animal_shelter import AnimalShelter, I will use:
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Database credentials are loaded securely from environment variables
# by the AnimalShelter class in CRUD_Python_Module.py

# Connect to database via CRUD Module
# Since it is already hardcoded in CRUD module, the constructor do not accept
# parameters (not need to pass them in from the dashboard). Instead of using 
# db = AnimalShelter(username, password), I will need to use:
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned.
# Added only load up to 100 animals compared to 6,000 at start up (load the other data later),
# and this prevents it from crashing. 
df = pd.DataFrame.from_records(db.read({})).head(100)

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIXED Grazioso Salvare’s logo
# instead of 'my-image.png', I added in Grazioso Salvare's logo png.
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIXED by placing the HTML image tag in the line below into the app.layout code according to my design
#FIXED to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

# The html.Div is the main container for the whole dashboard; becomes
# becomes part of the webpage layout thats inside the list. 
# for header (logo and title)
app.layout = html.Div([ 
    # group together the radio buttons.
    html.Div(
        style={
            'display': 'flex',       # placed in horizontal row.
            'alignItems': 'center',  # vertically centers.
            'justifyContent': 'space-between', # spreads left and center content evenly.
            'padding': '10px 20px'   # clean layout by adding spacing around the header.
        }, 
        children=[
            #On the left side, input the Grazioso Salvare Logo
            html.Img(
                src='data:image/png;base64,{}'.format(encoded_image.decode()),
                style={'height': '150px',      # logo will be professional and consistent sized. 
                       'marginRight': '20px'} # Add spacing so the logo does not crowd the text. 
            ),
            #Dashboard title and name centered
            html.Div(
                style={'textAlign': 'center',    # in this block will keep the title with name centered.
                       'flex': '1'},             # Expand the block to fill available space. 
                children=[
                    html.H1('CS-340 Dashboard'), # dashboard title
                    html.H3('Dashboard created by Natalie Fenton') #my identifier
                ]
            ),
            # Right side spacer to keep the website balanced/even. 
            # without the spacer, the center text would shift left. 
            html.Div(style={'width': '80px'})
            
        ]
    ),
     
   # add spacing under the logo
    html.Br(), 
    
   # Inputted another separator line to keep the layout organized. 
    html.Hr(),
    
        
#FIXED by adding in code for the interactive filtering options like
# Rescue Filter Buttons. 
# Neeed to be inside layout and before the DataTable so users can filter the table.
# And, the callback that updates the DataTable listens to 'filter-type', so it needs
# to exit before the table is rendered. 
    html.Div(  #Group he radio buttons together
        id='filter-options',
        style={
            'textAlign': 'left', 
            'padding': '10px 20px',
            'marginBottom': '10px'},
        
        children=[
            dcc.RadioItems(   
                id='filter-type', #very important (id), the callback will listen to this value
                                  #and update the DataTable based on what rescue type is selected.
            #Note: Each option makes one radio button for the user to click on. 
            # label is what the user sees.
            # value is what the callback receives.
                options=[ # all are filters to that selected option.
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain Rescue', 'value': 'mountain'},
                    {'label': 'Disaster Rescue', 'value': 'disaster'},
                    {'label': 'Reset', 'value': 'reset'} # shows all animals again (no filtering).
                ],
                value='reset', #default selection when the dashboard loads; all animals visible.
                inline=True,    #Makes radio buttons appear side-by-side horizontally (not vertically stacked).
                style={
                    'display': 'flex',
                    'gap': '20px', 
                    'fontSize': '18px'}
            )
        ]
    ),
    #separator for keeping organized.              
    html.Hr(),

#FIXED: Features for the interactive data table (user-friendly)
# Used resources from Module Six Assignment and copied here. 
    #Gives the DataTable a unique ID so that callbacks are able to update it.                
    dash_table.DataTable(
        id='datatable-id',
        # Makes a column for every field in the MongoDB Dataset
        # Note: deletable: false prevents users from deleting the column
        columns=[{"name": i, "id": i, "deletable": False,"selectable": True} 
                 for i in df.columns # able to highlight columns when clicked.
        ],
        #Loads the actual data (MongoDB) into the table. 
        #df.to)dict are records and are converted in the DataFrame into a list of dictionaries. 
        data=df.to_dict('records'),
        
        # features from module six
        #Settings for selection in which the user can choose a row and lets the dashboard
        # know which animal the user is looking at. 
        row_selectable='single', #selects one row at a time and is needed for map callback
        selected_rows=[0],       #first row is automatically selected when the dashboard loads
                                 # so that the map does not break when starting up (adds a parameter).
        
        #Table usability setting so that the table is user-friendly due to the huge AAC dataset.
        #Has options limited to 10 shown, sort out, and be able to type in the filters. 
        page_size=10,              # Only 10 rows will be shown at a time to prevent too much shown.
        sort_action="native",      # A built-in sorting for user by clicking column headers.
        filter_action="native",    # A built-in filtering for users to type that is located above columns.
        
        style_data_conditional=[], #callback will be able to update row highlighting.
        
        style_as_list_view=False, #make highlighting visiable
        
        style_cell={'padding': '5px'}, #highlight row to appear
    ),
    html.Br(), #adds a small vertical space
    html.Hr(), # Another separator due to the charts for space
                   
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display' : 'flex',
                'gap': '20px',           #place space between the map and graph
                'padding': '20px',       #input spacing inside the container
                'backgroundColor': '#f7f7f7',  #light background to stick out selected
                'borderRadius': '10px',  #input rounded corners
                'alignItems': 'stretch', #forces both columns to be same height
                'flexWrap': 'wrap'       # makes allows chart and map to stack on small screens
        },
        #children of this row: Pie chart and Map
        #Note: Both must be in the same flex row (side-by-side)
        children=[
            
            # Pie chart container
            html.Div(
                id='graph-id',
                style={'flex': '1',                 #column takes up only equal space
                       'backgroundColor': 'white',  #For constrast with black text
                       'padding': '10px',           #space size inside the chart container
                       'borderRadius': '10px',      #gives round corners, polished look
                       'minWidth': '450px',         #prevents chart from shrinking too small
                       'minHeight': '450px',        # prevent graph from collapsing
                       'maxWidth': '4500px',         # makes sure the chart does not stretch too wide 
                       'overflow': 'hidden',    #makes Jupyter not re-render the graph during layout shifts.
                       
                       'display': 'flex',
                       'justifyContent': 'center',
                       'alignItems': 'center'}
            ),
            # Map container
            # Note: midHeight makes sure the map does not collapse or disappear at certain zoom lvls.
            #  In addition, width: 100% makes the map fill the column properly (not look uneven). 
            html.Div(
                id='map-id',
                style={'flex': '1',                 
                       'backgroundColor': 'white', 
                       'padding': '10px', 
                       'borderRadius': '10px',
                       'minHeight': '450px',        # Makes sure map has enough vertical space
                       'maxHeight':'450',           # prevents stretching.
                       'width': '100%',             # Prevents problems with shrinking and layout stays stable
                       'flexShrink': 0,             # prevents map from collasping
                       'minWidth': '300px',         # makes sure that the map always stay visiable.
                       'overflow': 'hidden'}        #m akes Jupyter ot reflow crash.
            )
        ]
    )
])

#############################################
# Interaction Between Components / Controller
#############################################
    
@app.callback(Output('datatable-id','data'),  # the Dash will update the DataTable's rows.
              [Input('filter-type', 'value')]) # will listen bsed on what rescue filter button was clicked on.
                   
# function will run every time the user clicks on the filter button:
# water, mountain, disaster, or reset. 
def update_dashboard(filter_type):
    
    #Referencing from Grazioso Salvare table for each row. 
    #Water Rescue- Row 1.
    if filter_type == 'water':
        query = {"$and": [
            # Preferred Breeds.
            {"breed": {"$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retrieve",
                "Newfoundland"
            ]}},
            # Preferred Sex.
            {"sex_upon_outcome": "Intact Female"},
            # Training age preferred.
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
        ]}
        # Will get filtered data from MongoDB using CRUD Module           
        data = db.read(query)
                   
    #Mountain Rescue- Row 2.
    elif filter_type == 'mountain':
        query = {"$and": [
            # Preferred breeds.
            {"breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]}},
            # Preferred Sex.
            {"sex_upon_outcome": "Intact Male"},
            # Training age preferred
            {"age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}}
        ]}
        data = db.read(query)
                   
    #Disaster Rescue- Row 3.
    elif filter_type == 'disaster':
        query = {"$and": [
            #Preferred Breed.
            {"breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd", 
                "Golden Retriever",
                "Bloodhound"
                "Rottweiler"
            ]}},
            #Preferred Sex.
            {"sex_upon_outcome": "Intact Male"},
            #Training age preferred.
            {"age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}}
        ]}
        data = db.read(query) 
    
    # Reset (all animals shown).
    # However, needs to be limited otherwise it would crash and never load. 
    else: 
        data = db.read({})[:500] #Empty query returns the first 500 to prevent callback overload.
                   
    df_filtered = pd.DataFrame.from_records(data) #convert MongoDB results to DataTable Format.
                   
    # Removes MongoDb's _id field since Dash cannot display ObjectId. 
    if '_id' in df_filtered.columns:
        df_filtered.drop(columns=['_id'], inplace=True)
    # Return filtered data to the DataTable.   
    return df_filtered.to_dict('records')
                   
# Display the breeds of animal based on quantity represented in
# the data table.
@app.callback(
    Output('graph-id', "children"), #updates whats in inside graph-id <div>.
    [Input('datatable-id', "derived_virtual_data")]) #takes visible DataTable rows as input.
# filtered table rows(list of dicts = viewData.
def update_graphs(viewData):
    
    #Preventing early callback crashes (since it would not load staring program for reset)
    if viewData is None: 
        return []
    
    dff = pd.DataFrame(viewData) #conversion from table rows to DataFram for charting.
                                # Ploty charts require structured data. 
        
    dff = dff.dropna(subset=['breed']) #Handles none vaules in the breed column. This will drop the 
                                       # with missing breed values the prevent the graph from freezing. 
                               
    if 'breed' not in dff.columns: #if breed column disappears this will 
        return []                  #prevent errors when trying to chart missing data.
    
    #Need to group the breeds so that it is faster and more organized.
    #Note: what it does it is counts how many times each breed appears in the filtered dataset. 
    #      THis will create a summary table with two columns: breed and count. 
    dff_grouped = dff.groupby('breed').size().reset_index(name='count')
    
    threshold = 5 #Need to have the other breeds be 'other' to prevent the pie chart from being unreadable.
    
    dff_grouped['breed_grouped'] = dff_grouped.apply(
        lambda row: row['breed'] if row ['count'] >= threshold else 'Other',
        axis=1
    )
    #regroup using the new category
    dff_grouped = dff_grouped.groupby('breed_grouped')['count'].sum().reset_index()
                        
                        
    #build a pie chart using grouped data.
    fig = px.pie(                  
        dff_grouped,                 #filtered DataFrame
        names='breed_grouped',             #by name be able to group into slices
        values='count',            #slice size is number of animals of that breed.
        title='Breed Distribution' #Chart title 
    )
    
    #This fixes the shrinking when only one breed exists (reset to Disaster Rescue, the Disaster rescue pie chart 
    #shrinks.Controlls only the SVG size, not the container
    fig.update_layout(
        height=450,     #force the chart to be 400px tall
        width=450,      #force the chart to be 400px wide
        autosize=False, # makes the Ploty not auto resize the SVG
        margin=dict(    # to prevent touching edges by adding padding around the chart.
            l=20,  # left margin
            r=20,  # Right margin
            t=40,  # Top margin
            b=20),  # Bottom margin
        
        paper_bgcolor='white', # background is kept clean
        plot_bgcolor='white',  #
        title_x=0.5,   # keeps the pie title centered (Note: 0=left, 1=right, 0.5=centered).
        
        legend=dict(   #right side legend
            orientation='v', #vertical legend
            x=1.2,          # move legend to the right of chart
            y=0.5,           # center the legend vertically
            xanchor='left',  # anchor legend from its left edge
            yanchor='middle') # anchor legend from its middle
    )
    
    #center pie chart domain
    fig.update_traces(
        textposition='outside', # percentages inside slices
    )
     
    return [dcc.Graph(figure=fig)] #return the chart wrapped in a Graphy component ( and return it as a list). 
                                    #Note: Dash needs children to be returned as a list (not raw figures).

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_rows')]
)
def update_styles(selected_rows):
    
    if not selected_rows: #if empty list, return so prevent crashing.
        return []
    
    return [{
        'if': {'row_index': selected_rows[0]},
        'background_color': '#FFF2A8', #bright gold highlight
        'color': 'black',              
        'fontWeight': 'bold',          #make row pop out
        'border': '2px solid #E0A800'  #thick border around the row
    }]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),     #updates the map display area.
    [Input('datatable-id', "derived_virtual_data"), #in the DataTable all rows are visiable.
     Input('datatable-id', "derived_virtual_selected_rows")]) #selected row index. 
def update_map(viewData, index): 
    
    # Need to prevent early callback crashes (Reset loading keeps having errors)
    # the mapcall back before was firing too early
    if viewData is None or index is None:
        return []
    
    # Need to mapcall back stable and secure against missing columns and selections.
    if not viewData: # If table has not row (empty after filtering) 
        return []    # returns an empty map area to prevent crashing.
                               
    if not index: #If user has not selected a row yet, it will avoid errors from trying to 
        return [] #access a missing row. 
    
    dff = pd.DataFrame.from_dict(viewData) #Convert the DataTable rows to DataFrame for easy access. 
  
    #When the latitude or longtitude columns missing, this is to halt to avoid map errors. 
    if 'location_lat' not in dff.columns or 'location_long' not in dff.columns:
        return []
                               
    row = index[0] #Only one row can be selected to be able to extract the index number.
    
    #if chosen row has any mising latitude or longitude, this will stop and return an empty map. 
    #Note: some of the AAC animals might not have a GPS data, and we need to check to prevent it
    # from crashing and return an empty map instead. 
    if pd.isna(dff.iloc[row]['location_lat']) or pd.isna(dff.iloc[row]['location_long']):
        return [] 
        
    # Austin TX is at [30.75,-97.48]
    # Needed to do 100% to fill entire right column and prevent map collapse
    return [
        dl.Map(style={'width': '100%', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://escapechris-charmdialog-3000.codio.io/proxy/8050/
